In [0]:
import os, sys, yaml
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
with open(os.path.join(common.repo_root(), "config", "group.yml")) as f:
    grp = yaml.safe_load(f)
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

src = grp["source"][env]
print(cat, "reads", f"{src}.gold")

In [0]:
check("this is the BHG repo (group runs in the parent only)", cfg["code"] == "bhg", cfg["code"])
check("FQF source is the same environment", src == f"fqf_{env}", src)

need = ["dim_customer", "dim_loan", "dim_product", "dim_branch", "fact_fqf_event"]
for t in need:
    try:
        n = spark.table(f"{src}.gold.{t}").count()
        check(f"can read {src}.gold.{t}", n > 0, n)
    except Exception as e:
        check(f"can read {src}.gold.{t}", False, str(e).split("\n")[0][:140])

codes = [c["code"] for c in grp["companies"]]
check("two companies listed, one parent", codes == ["bhg", "fqf"]
      and [c["role"] for c in grp["companies"]].count("parent") == 1, codes)

bhg_types = {r[0] for r in spark.table(f"{cat}.gold.dim_payer").select("payer_type").distinct().collect()}
check("BHG intercompany payer type exists", grp["intercompany"]["bhg_payer_type"] in bhg_types, bhg_types)
links = spark.table(f"{src}.gold.fact_fqf_event").filter("event_type = 'disbursement' AND is_hospital_bill").count()
check("FQF Gold has hospital-bill disbursements", links > 0, links)
print("INFO hospital-bill disbursements in", src, ":", links)

print(f"\nt21_group_access: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t21_group_access failed")